In [11]:
import json
from pathlib import Path
from typing import Any, Optional, Callable


import matplotlib.pyplot as plt
import torch
from jaxtyping import Float
import pandas as pd
import tqdm
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State, callback_context

# custom utils
from muutils.spinner import SpinnerContext
from muutils.jsonlines import jsonl_load, jsonl_write

# pattern_lens
from pattern_lens.consts import (
	SPINNER_KWARGS,
)
from pattern_lens.load_activations import load_activations
from pattern_lens.figures import HTConfigMock

In [ ]:
def scalar_features(
	features_func: Callable[
		[Float[torch.Tensor, "batch n_ctx n_ctx"]],
		dict[str, Float[torch.Tensor, " batch"]],
	],
	save_path: Path = Path("../docs/temp"),
	models: list[str] | None = None,
) -> pd.DataFrame:
	if models is None:
		models = [
			json.loads(cfg)["model_name"]
			for cfg in (save_path / "models.jsonl").read_text().splitlines()
		]

	print(f"models: {models}")

	# output has cols:
	# model, prompt, layer_idx, head_idx, feature_name, feature_value
	output: list[dict] = list()

	for idx, model in enumerate(models):
		print(f"model: '{model}'")
		with SpinnerContext(message="setting up paths", **SPINNER_KWARGS):
			model_path: Path = save_path / model
			with open(model_path / "model_cfg.json", "r") as f:
				model_cfg = HTConfigMock.load(json.load(f))

		with SpinnerContext(message="loading prompts", **SPINNER_KWARGS):
			# load prompts
			with open(model_path / "prompts.jsonl", "r") as f:
				prompts: list[dict] = [json.loads(line) for line in f.readlines()]
			# truncate to n_samples
			prompts = prompts

		print(f"{len(prompts)} prompts loaded")

		for prompt in tqdm.tqdm(prompts, desc="prompts", total=len(prompts)):
			activations_path, cache = load_activations(
				model_name=model_cfg.model_name,
				prompt=prompt,
				save_path=save_path,
				return_fmt="numpy",
			)

			for key, head_batch in cache.items():
				features = features_func(torch.tensor(head_batch[0]))
				for feature_name, feature_value in features.items():
					for head_idx in range(feature_value.shape[0]):
						output.append(
							dict(
								model=model,
								prompt=prompt["hash"],
								layer=idx,
								head=head_idx,
								feat_name=feature_name,
								feat_val=feature_value[head_idx].item(),
							)
						)

	return pd.DataFrame(output)

In [ ]:
def compute_scalar_features(
	attn_batch: Float[torch.Tensor, "batch n_ctx n_ctx"],
) -> dict[str, Float[torch.Tensor, " batch"]]:
	diag_weight: Float[torch.Tensor, " batch"] = attn_batch.diagonal(
		0, dim1=-2, dim2=-1
	).mean(dim=-1)
	first_token_weight: Float[torch.Tensor, " batch"] = attn_batch[:, 0, :].mean(dim=-1)

	return dict(
		diag_weight=diag_weight,
		first_token_weight=first_token_weight,
	)


df = scalar_features(features_func=compute_scalar_features)

models: ['pythia-14m', 'gemma-2b', 'gpt2-small', 'pythia-1b', 'gpt2-medium', 'tiny-stories-1M']
model: 'pythia-14m'
✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
128 prompts loaded


prompts: 100%|██████████| 128/128 [00:01<00:00, 76.04it/s]

model: 'gemma-2b'
| (0.00s) setting up paths                                                     

✔️  (0.00s) setting up paths                                                   
✔️  (0.00s) loading prompts                                                    
128 prompts loaded


prompts: 100%|██████████| 128/128 [00:07<00:00, 17.99it/s]

...

✔️  (0.00s) loading prompts                                                    
128 prompts loaded


prompts: 100%|██████████| 128/128 [00:22<00:00,  5.57it/s]

model: 'tiny-stories-1M'
✔️  (0.00s) setting up paths                                                   
✔️  (0.00s) loading prompts                                                    


128 prompts loaded


prompts: 100%|██████████| 128/128 [00:08<00:00, 15.84it/s]


In [ ]:
jsonl_write(
	"../data/scalar_features.jsonl.gz", df.to_dict(orient="records"), use_gzip=True
)

In [10]:
df = jsonl_load("../data/scalar_features.jsonl.gz")

In [ ]:
def plot_histograms_long(df: pd.DataFrame) -> None:
	"""Plot histograms for each feature with different models superimposed.

	This function assumes the DataFrame is in long format with columns:
	"model", "feat_name", "feat_val", and optionally "prompt", "layer", "head".

	# Parameters:
	 - `df : pd.DataFrame`
	     DataFrame containing the data.

	# Returns:
	 - `None`
	     Displays the histograms.
	"""
	features: list[str] = df["feat_name"].unique().tolist()
	models: list[str] = df["model"].unique().tolist()

	for feature in features:
		plt.figure()
		subset_feature: pd.DataFrame = df[df["feat_name"] == feature]
		for model in models:
			subset_model: pd.DataFrame = subset_feature[
				subset_feature["model"] == model
			]
			plt.hist(
				subset_model["feat_val"], bins=50, alpha=0.5, label=model, density=True
			)
		plt.xlabel(feature)
		plt.ylabel("Frequency")
		plt.title(f"Histogram of {feature} for different models")
		plt.legend()
		plt.show()


plot_histograms_long(df)

In [ ]:
df = pd.read_json("scalar_features.jsonl", orient="records", lines=True)

In [ ]:
def pivot_features(df: pd.DataFrame) -> pd.DataFrame:
	"""Pivot the DataFrame so that each head (grouped by model, prompt, layer, head)
	has its features as columns.

	# Parameters:
	 - `df : pd.DataFrame`
	    DataFrame with columns including "model", "prompt", "layer", "head", "feat_name", "feat_val".

	# Returns:
	 - `pd.DataFrame`
	    Pivoted DataFrame with one column per feature.
	"""
	pivot_df: pd.DataFrame = df.pivot_table(
		index=["model", "prompt", "layer", "head"],
		columns="feat_name",
		values="feat_val",
	).reset_index()
	return pivot_df


def create_scatter_figure(
	df: pd.DataFrame,
	x_feature: str,
	y_feature: str,
	selected_layer: Optional[int] = None,
	selected_head: Optional[int] = None,
) -> Any:
	"""Create a Plotly scatter plot figure with the given features.

	If a head is selected, non-selected points are drawn with low opacity instead of being removed.

	# Parameters:
	 - `df : pd.DataFrame`
	    Pivoted DataFrame with one row per head and columns for each feature.
	 - `x_feature : str`
	    Feature name to use for the x-axis.
	 - `y_feature : str`
	    Feature name to use for the y-axis.
	 - `selected_layer : Optional[int]`
	    Layer to highlight. Other points will be drawn with low alpha.
	 - `selected_head : Optional[int]`
	    Head to highlight. Other points will be drawn with low alpha.

	# Returns:
	 - `plotly.graph_objects.Figure`
	    The scatter plot figure.
	"""
	# Copy data and compute base opacity based on layer.
	df_plot: pd.DataFrame = df.copy()
	min_layer: int = int(df_plot["layer"].min())
	max_layer: int = int(df_plot["layer"].max())
	if max_layer == min_layer:
		df_plot["base_opacity"] = 1.0
	else:
		df_plot["base_opacity"] = 0.5 + 0.5 * (
			(df_plot["layer"] - min_layer) / (max_layer - min_layer)
		)

	# Determine final marker opacity: if a selection exists, then for points that don't match, use a low opacity.
	def compute_opacity(row: pd.Series) -> float:
		if selected_layer is not None and selected_head is not None:
			if (row["layer"] == selected_layer) and (row["head"] == selected_head):
				return row["base_opacity"]
			else:
				return 0.1  # low opacity for non-selected points
		return row["base_opacity"]

	df_plot["opacity"] = df_plot.apply(compute_opacity, axis=1)

	# Create scatter plot.
	fig: Any = px.scatter(
		df_plot,
		x=x_feature,
		y=y_feature,
		color="model",
		custom_data=["model", "layer", "head"],
		title=f"Scatterplot of {x_feature} vs {y_feature}",
	)
	fig.update_traces(marker=dict(opacity=df_plot["opacity"]))
	fig.update_traces(
		hovertemplate="<br>".join(
			[
				"Model: %{customdata[0]}",
				"Layer: %{customdata[1]}",
				"Head: %{customdata[2]}",
				f"{x_feature}: " + "%{x:.3f}",
				f"{y_feature}: " + "%{y:.3f}",
				"<extra></extra>",
			]
		)
	)
	return fig


# Dash App Setup
app: Dash = Dash(__name__)
server: Any = app.server  # expose server for deployment

# Load and pivot the data (replace 'data.jsonl' with your JSONL filepath)
df_raw: pd.DataFrame = df
df_pivot: pd.DataFrame = pivot_features(df_raw)

# For now, assume we have exactly two features.
X_FEATURE: str = "diag_weight"
Y_FEATURE: str = "first_token_weight"

app.layout = html.Div(
	[
		html.H1("Feature Consistency Scatterplot"),
		dcc.Graph(
			id="scatter-plot",
			figure=create_scatter_figure(df_pivot, X_FEATURE, Y_FEATURE),
			config={"displayModeBar": True},
		),
		html.Button("Clear selection", id="clear-button", n_clicks=0),
		dcc.Store(id="filtered-state", data=None),
	]
)


@app.callback(
	Output("scatter-plot", "figure"),
	Output("filtered-state", "data"),
	Input("scatter-plot", "clickData"),
	Input("clear-button", "n_clicks"),
	State("filtered-state", "data"),
)  # type: ignore
def update_scatter(
	click_data: Any, clear_clicks: int, stored_state: Optional[dict[str, Any]]
) -> tuple[Any, Optional[dict[str, Any]]]:
	"""Update the scatter plot based on click events and clear button.

	If a point is clicked, the plot will highlight points corresponding to that head and layer,
	while other points are drawn with low opacity.
	Pressing the clear button resets the selection.

	# Parameters:
	 - `click_data : Any`
	    Data about the clicked point from the scatter plot.
	 - `clear_clicks : int`
	    The number of times the clear button was clicked.
	 - `stored_state : Optional[dict[str, Any]]`
	    Previously stored filter state.

	# Returns:
	 - `Tuple[Any, Optional[dict[str, Any]]]`
	    Updated figure and updated state.
	"""
	ctx: Any = callback_context
	triggered_id: str = ""
	if ctx.triggered:
		triggered_id = ctx.triggered[0]["prop_id"].split(".")[0]

	if triggered_id == "clear-button":
		return create_scatter_figure(df_pivot, X_FEATURE, Y_FEATURE), None

	if click_data is not None:
		point: dict[str, Any] = click_data["points"][0]
		layer_val: int = int(point["customdata"][1])
		head_val: int = int(point["customdata"][2])
		return create_scatter_figure(
			df_pivot,
			X_FEATURE,
			Y_FEATURE,
			selected_layer=layer_val,
			selected_head=head_val,
		), {"layer": layer_val, "head": head_val}

	return create_scatter_figure(df_pivot, X_FEATURE, Y_FEATURE), stored_state


app.run(debug=True)